### Volunteer Scheduling - Goal Programming

In [ ]:
import pandas as pd
import pulp
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# read in qualtrics export (first 2 rows are metadata, skip them)
pref_df = pd.read_csv('Volunteer Availability.csv', skiprows=[0,1])

new_header = ['Name', 'Email', 'Week 1', 'Week 2', 'Week 3', 'Week 4', 'Week 5', 'Week 6', 'Week 7', 'Confirmed by volunteer']
pref_df.columns = new_header
pref_df.insert(0, 'Volunteer ID', pref_df.index + 1)

# pref_map keeps prefer vs available distinct (2 vs 1) for the objective
# avail_map collapses to binary for the constraints
pref_map = {
    'I am available and prefer this week': 2,
    'I am available': 1,
    'I am not available': 0
}
avail_map = {
    'I am available and prefer this week': 1,
    'I am available': 1,
    'I am not available': 0
}

availability_df = pref_df.copy()
for col in pref_df.columns[3:]:
    pref_df[col] = pref_df[col].map(pref_map)
for col in availability_df.columns[3:]:
    availability_df[col] = availability_df[col].map(avail_map)

# how many volunteers each week needs
weekly_volunteers = [3, 2, 4, 5, 4, 3, 2]

volunteers = availability_df['Volunteer ID'].tolist()
weeks = list(range(1, 8))
week_cols = availability_df.columns[3:]

# drop anyone who marked unavailable for every week - they would break the model
zero_avail = availability_df[availability_df[week_cols].sum(axis=1) == 0]
if not zero_avail.empty:
    print('WARNING: removed volunteers with no availability:')
    for name in zero_avail['Name'].tolist():
        print(f'  - {name}')
    availability_df = availability_df[availability_df[week_cols].sum(axis=1) > 0].reset_index(drop=True)
    pref_df = pref_df[pref_df['Volunteer ID'].isin(availability_df['Volunteer ID'])].reset_index(drop=True)
    volunteers = availability_df['Volunteer ID'].tolist()

availability_df

### Goal Programming Model

In [ ]:
model = pulp.LpProblem('Volunteer_Scheduling', pulp.LpMinimize)

# x[i,j] = 1 if volunteer i goes to week j
x = pulp.LpVariable.dicts('x', [(i, j) for i in volunteers for j in weeks], cat='Binary')

# slack for headcount - absorbs shortfall/excess instead of crashing
d_under = pulp.LpVariable.dicts('d_under', weeks, lowBound=0)
d_over  = pulp.LpVariable.dicts('d_over',  weeks, lowBound=0)

# binary - did this volunteer go unassigned?
unassigned = pulp.LpVariable.dicts('unassigned', volunteers, cat='Binary')

# fires when someone is assigned to a week they can do but didn't prefer
pref_miss = pulp.LpVariable.dicts('pref_miss', [(i, j) for i in volunteers for j in weeks], lowBound=0)

# volunteers with fewer available weeks get a higher weight so they are prioritized
availability_counts = {
    i: availability_df.loc[availability_df['Volunteer ID'] == i, availability_df.columns[3:]]
       .gt(0).sum(axis=1).values[0]
    for i in volunteers
}
max_avail = max(availability_counts.values())
min_avail = min(availability_counts.values())
weights = {
    i: (max_avail - availability_counts[i]) / (max_avail - min_avail + 1)
    for i in volunteers
}

# each week hits its headcount target (slack catches any gap)
for j in weeks:
    model += (
        pulp.lpSum(x[(i,j)] * availability_df.iloc[i-1, j+2] for i in volunteers)
        + d_under[j] - d_over[j] == weekly_volunteers[j-1],
        f'Week_{j}_headcount'
    )

# everyone gets at least one assignment
for i in volunteers:
    model += (
        pulp.lpSum(x[(i,j)] * availability_df.iloc[i-1, j+2] for j in weeks)
        + unassigned[i] >= 1,
        f'Vol_{i}_assigned'
    )

# track when someone lands on a non-preferred week
for i in volunteers:
    for j in weeks:
        avail = availability_df.iloc[i-1, j+2]
        pref  = pref_df.iloc[i-1, j+2]
        not_preferred = 1 if (avail == 1 and pref == 1) else 0
        model += (pref_miss[(i,j)] >= x[(i,j)] * not_preferred, f'PrefMiss_{i}_{j}')

# penalty weights in priority order
# understaffing is the worst outcome so it gets double-penalized vs overstaffing
P1 = 1000  # fill every week
P2 = 100   # assign everyone
P3 = 10    # respect preferences
P4 = 1     # protect less flexible volunteers

model += (
    P1 * pulp.lpSum(2*d_under[j] + d_over[j] for j in weeks)
    + P2 * pulp.lpSum(unassigned[i] for i in volunteers)
    + P3 * pulp.lpSum(pref_miss[(i,j)] for i in volunteers for j in weeks)
    - P4 * pulp.lpSum(weights[i] * x[(i,j)] for i in volunteers for j in weeks)
)

model.solve(pulp.PULP_CBC_CMD(msg=0))

print('Status:', pulp.LpStatus[model.status])

print('\nWeekly staffing:')
for j in weeks:
    under    = pulp.value(d_under[j])
    over     = pulp.value(d_over[j])
    assigned = int(weekly_volunteers[j-1] - under + over)
    if under > 0.5:
        print(f'  Week {j}: short by {int(round(under))} ({assigned} assigned)')
    elif over > 0.5:
        print(f'  Week {j}: over by {int(round(over))} ({assigned} assigned)')
    else:
        print(f'  Week {j}: OK ({weekly_volunteers[j-1]} assigned)')

print('\nPlacements:')
for i in volunteers:
    name = pref_df.loc[pref_df['Volunteer ID'] == i, 'Name'].values[0]
    if pulp.value(unassigned[i]) > 0.5:
        print(f'  {name}: not placed')
    else:
        placed = [j for j in weeks if pulp.value(x[(i,j)] * availability_df.iloc[i-1, j+2]) > 0.5]
        print(f'  {name}: week(s) {placed}')

### Export to Excel

In [ ]:
wb = Workbook()

# schedule sheet
ws = wb.active
ws.title = 'Schedule'
ws.sheet_properties.tabColor = 'daa2fc'

cell_mapping = [('A','B'), ('C','D'), ('E','F'), ('G','H'), ('I','J'), ('K','L'), ('M','N')]
header_font  = Font(bold=True, name='Arial')
header_fill  = PatternFill(start_color='daa2fc', end_color='daa2fc', fill_type='solid')
short_fill   = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')
cell_font    = Font(name='Arial')

week_number = 1
for start_col, end_col in cell_mapping:
    ws.merge_cells(f'{start_col}1:{end_col}1')
    cell = ws[f'{start_col}1']
    cell.font      = header_font
    cell.alignment = Alignment(horizontal='center')

    short = pulp.value(d_under[week_number])
    if short > 0.5:
        cell.value = f'Week {week_number} \u26a0 SHORT {int(round(short))}'
        cell.fill  = short_fill
    else:
        cell.value = f'Week {week_number}'
        cell.fill  = header_fill

    row = 2
    for i in volunteers:
        if pulp.value(x[i, week_number] * pref_df.iloc[i-1, week_number+2]) > 0.5:
            ws[f'{start_col}{row}'].value = pref_df.loc[pref_df['Volunteer ID'] == i, 'Name'].values[0]
            ws[f'{end_col}{row}'].value   = pref_df.loc[pref_df['Volunteer ID'] == i, 'Email'].values[0]
            ws[f'{start_col}{row}'].font  = cell_font
            ws[f'{end_col}{row}'].font    = cell_font
            row += 1
    week_number += 1

for col_idx in range(1, 15):
    ws.column_dimensions[get_column_letter(col_idx)].width = 22

# diagnostics sheet
wd = wb.create_sheet(title='Diagnostics')
wd.sheet_properties.tabColor = 'FF0000'

title_font   = Font(bold=True, size=13, name='Arial')
section_font = Font(bold=True, size=11, name='Arial', color='FFFFFF')
section_fill = PatternFill(start_color='5a5a5a', end_color='5a5a5a', fill_type='solid')
ok_fill      = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
warn_fill    = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')
error_fill   = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
ok_font      = Font(name='Arial', color='276221')
warn_font    = Font(name='Arial', color='9C6500')
error_font   = Font(name='Arial', color='9C0006')

def write_cell(sheet, row, col, value, font=None, fill=None, alignment=None):
    c = sheet.cell(row=row, column=col, value=value)
    if font:      c.font = font
    if fill:      c.fill = fill
    if alignment: c.alignment = alignment
    return c

write_cell(wd, 1, 1, 'Schedule Diagnostics', font=title_font)
wd.column_dimensions['A'].width = 30
wd.column_dimensions['B'].width = 18
wd.column_dimensions['C'].width = 40

cur = 3

write_cell(wd, cur, 1, 'Weekly Staffing Summary', font=section_font, fill=section_fill)
write_cell(wd, cur, 2, '', fill=section_fill)
write_cell(wd, cur, 3, '', fill=section_fill)
cur += 1
write_cell(wd, cur, 1, 'Week',     font=Font(bold=True, name='Arial'))
write_cell(wd, cur, 2, 'Assigned', font=Font(bold=True, name='Arial'))
write_cell(wd, cur, 3, 'Status',   font=Font(bold=True, name='Arial'))
cur += 1

for j in weeks:
    under    = pulp.value(d_under[j])
    over     = pulp.value(d_over[j])
    assigned = int(weekly_volunteers[j-1] - round(under) + round(over))
    if under > 0.5:
        f, fi, status = error_font, error_fill, f'SHORT by {int(round(under))}'
    elif over > 0.5:
        f, fi, status = warn_font, warn_fill, f'OVER by {int(round(over))}'
    else:
        f, fi, status = ok_font, ok_fill, 'OK'
    write_cell(wd, cur, 1, f'Week {j}', font=f, fill=fi)
    write_cell(wd, cur, 2, assigned,    font=f, fill=fi)
    write_cell(wd, cur, 3, status,      font=f, fill=fi)
    cur += 1

cur += 1

write_cell(wd, cur, 1, 'Volunteer Placements', font=section_font, fill=section_fill)
write_cell(wd, cur, 2, '', fill=section_fill)
write_cell(wd, cur, 3, '', fill=section_fill)
cur += 1
write_cell(wd, cur, 1, 'Volunteer',      font=Font(bold=True, name='Arial'))
write_cell(wd, cur, 2, 'Weeks Assigned', font=Font(bold=True, name='Arial'))
write_cell(wd, cur, 3, 'Status',         font=Font(bold=True, name='Arial'))
cur += 1

for i in volunteers:
    name = pref_df.loc[pref_df['Volunteer ID'] == i, 'Name'].values[0]
    placed_weeks = [f'Week {j}' for j in weeks
                    if pulp.value(x[(i,j)] * availability_df.iloc[i-1, j+2]) > 0.5]
    if pulp.value(unassigned[i]) > 0.5:
        f, fi, status, weeks_str = error_font, error_fill, 'NOT PLACED', '\u2014'
    else:
        f, fi, status, weeks_str = ok_font, ok_fill, 'Placed', ', '.join(placed_weeks)
    write_cell(wd, cur, 1, name,      font=f, fill=fi)
    write_cell(wd, cur, 2, weeks_str, font=f, fill=fi)
    write_cell(wd, cur, 3, status,    font=f, fill=fi)
    cur += 1

# only show this section if someone was dropped for having no availability
if not zero_avail.empty:
    cur += 1
    write_cell(wd, cur, 1, 'Removed (No Availability)', font=section_font, fill=section_fill)
    write_cell(wd, cur, 2, '', fill=section_fill)
    write_cell(wd, cur, 3, '', fill=section_fill)
    cur += 1
    for name in zero_avail['Name'].tolist():
        write_cell(wd, cur, 1, name,      font=error_font, fill=error_fill)
        write_cell(wd, cur, 2, 'Removed', font=error_font, fill=error_fill)
        write_cell(wd, cur, 3, 'Marked unavailable for all weeks', font=error_font, fill=error_fill)
        cur += 1

wb.save('Volunteer_Schedule.xlsx')
print('Saved: Volunteer_Schedule.xlsx')